# ProtSpace: Data Preparation

Prepare protein data for interactive visualization at [protspace.app](https://protspace.app/explore).

**Three input modes:**
- **Example H5** — download pre-computed ProtT5 embeddings to try the pipeline
- **Upload file** — bring your own HDF5 embeddings or FASTA sequences
- **UniProt query** — fetch proteins directly from UniProt

📚 [GitHub](https://github.com/tsenoner/protspace) · [Manuscript](https://doi.org/10.1016/j.jmb.2025.168940) · [CLI Reference](https://github.com/tsenoner/protspace/blob/main/docs/cli.md)


In [ ]:
# @title 1. Install & Setup (~30s)
%%capture
!pip install -qqq protspace

import urllib.request
from pathlib import Path

import h5py
import ipywidgets as widgets

# Patch tqdm for notebook-style progress bars BEFORE importing protspace
import tqdm as _tqdm_mod
import tqdm.notebook as _tqdm_nb
from google.colab import files
from IPython.display import HTML as IHTML
from IPython.display import clear_output, display

_tqdm_mod.tqdm = _tqdm_nb.tqdm

# Pre-import protspace (lightweight now — heavy reducers load per-method)
import time as _time

import pandas as _pd

from protspace.data.loaders import embed_fasta, load_h5
from protspace.data.loaders.query import query_uniprot
from protspace.data.processors.pipeline import (
    PipelineConfig,
    ReducerParams,
    ReductionPipeline,
    parse_methods_arg,
)

In [ ]:
# @title 2. Choose Input Data {display-mode: "form"}
# @markdown Run this cell, then select a tab and follow the instructions.

# Fix Tab label color in Colab dark mode.
# Colab's built-in fix (colabtools#1895) bridges --jp-ui-font-color1 but
# Tab labels use --jp-ui-font-color0, which falls back to near-black.
# This CSS must be in the SAME cell as the Tab widget (Colab uses per-cell iframes).
display(IHTML("""<style>
:root { --jp-ui-font-color0: var(--colab-primary-text-color, inherit); }
.p-TabBar-tab, .lm-TabBar-tab { color: inherit !important; }
.widget-html-content a { text-decoration: underline !important; }
</style>"""))

_RELEASE_URL = "https://github.com/tsenoner/protspace/releases/download/examples/"

_EXAMPLES = {
    "Three-Finger Toxin (536 proteins)": "three_finger_toxin_prot_t5.h5",
    "Beta-Lactamase (731 proteins)": "beta_lactamase_prot_t5.h5",
    "Globin (1,200 proteins)": "globin_prot_t5.h5",
    "Phosphatase (1,587 proteins)": "phosphatase_prot_t5.h5",
    "Snake Toxin incl. TrEMBL (5,015 proteins)": "snake_toxin_prot_t5.h5",
}

_QUERY_EXAMPLES = [
    '(family:"three-finger toxin") AND (reviewed:true)',
    '(family:globin) AND (reviewed:true)',
    '(family:phosphatase) AND (reviewed:true)',
    '(keyword:Toxin) AND (taxonomy_id:8570)',
]

ALL_EMBEDDERS = [
    "prot_t5", "prost_t5",
    "esm2_8m", "esm2_35m", "esm2_150m", "esm2_650m", "esm2_3b",
    "ankh_base", "ankh_large", "ankh3_large",
    "esmc_300m", "esmc_600m",
]

input_args = {}


def _validate_h5(filepath):
    with h5py.File(filepath, "r") as f:
        keys = list(f.keys())
        if not keys:
            raise ValueError("File is empty")
        item = f[keys[0]]
        if isinstance(item, h5py.Group):
            sub = list(item.keys())
            if sub:
                item = item[sub[0]]
        arr = item[:]
        if arr.ndim == 2 and arr.shape[0] == 1:
            arr = arr.squeeze(0)
        if arr.ndim > 1:
            raise ValueError(f"Per-residue embeddings (shape {item.shape}). Needs per-protein.")
        return len(keys), arr.shape[0]


# --- Embedder multi-select (shown for FASTA & UniProt query tabs) ---
emb_cbs = {}
for e in ALL_EMBEDDERS:
    emb_cbs[e] = widgets.Checkbox(
        value=(e == "prot_t5"), description=e, indent=False, layout={"width": "auto"},
    )
emb_grid = widgets.GridBox(
    list(emb_cbs.values()),
    layout={"grid_template_columns": "repeat(4, auto)", "grid_gap": "0 10px"},
)
sim_cb = widgets.Checkbox(value=False, description="Sequence similarity (MMseqs2)", indent=False)
embedder_section = widgets.VBox([
    widgets.HTML("<b>Select embedder(s)</b>"),
    emb_grid, sim_cb,
])
embedder_section.layout.display = "none"


def _get_selected_embedders():
    return [e for e, cb in emb_cbs.items() if cb.value]


# --- Tab 1: Example Dataset (auto-select on dropdown change) ---
example_dropdown = widgets.Dropdown(
    options=list(_EXAMPLES.keys()),
    description="Dataset:", style={"description_width": "initial"},
    layout={"width": "70%"},
)
example_output = widgets.Output()


def _select_example(name):
    """Download (if needed) and select an example dataset."""
    global input_args
    with example_output:
        clear_output()
        filename = _EXAMPLES[name]
        if not Path(filename).exists():
            print(f"Downloading {filename}...")
            try:
                urllib.request.urlretrieve(_RELEASE_URL + filename, filename)
            except Exception as e:
                print(f"Download failed: {e}")
                return
        try:
            n, dim = _validate_h5(filename)
            print(f"Selected: {name} ({n} proteins, {dim}D ProtT5 embeddings)")
            input_args = {"type": "h5", "path": filename, "name": "prot_t5"}
            embedder_section.layout.display = "none"
        except Exception as e:
            print(f"Validation failed: {e}")


def _on_example_change(change):
    if change["name"] != "value":
        return
    _select_example(change["new"])


example_dropdown.observe(_on_example_change, names="value")
tab_example = widgets.VBox([
    widgets.HTML("<b>Select a pre-computed ProtT5 dataset</b>"),
    example_dropdown,
    example_output,
])

# --- Tab 2: Upload Embeddings (.h5) ---
h5_upload = widgets.FileUpload(accept=".h5,.hdf5,.hdf,.gz", multiple=False)
h5_output = widgets.Output()


def _on_h5_upload(_change):
    global input_args
    if not h5_upload.value:
        return
    with h5_output:
        clear_output()
        fname = list(h5_upload.value.keys())[0]
        content = h5_upload.value[fname]["content"]
        Path(fname).write_bytes(content)
        if fname.endswith(".gz"):
            import gzip
            extracted = fname[:-3]
            with gzip.open(fname, "rb") as gz:
                Path(extracted).write_bytes(gz.read())
            fname = extracted
        try:
            n, dim = _validate_h5(fname)
            print(f"Ready: {fname} ({n} proteins, {dim}D)")
            input_args = {"type": "h5", "path": fname}
            embedder_section.layout.display = "none"
        except Exception as e:
            print(f"Validation failed: {e}")


h5_upload.observe(_on_h5_upload, names="value")
tab_h5 = widgets.VBox([
    widgets.HTML(
        "<b>Upload HDF5 embeddings</b><br>"
        "<i>Get ProtT5 embeddings from "
        '<a href="https://www.uniprot.org/" target="_blank">UniProt</a> '
        "or generate them with the "
        '<a href="https://colab.research.google.com/github/tsenoner/protspace/blob/main/'
        'notebooks/ClickThrough_GenerateEmbeddings.ipynb" target="_blank">Embeddings notebook</a></i>'
    ),
    h5_upload,
    h5_output,
])

# --- Tab 3: Upload FASTA ---
fasta_upload = widgets.FileUpload(accept=".fasta,.fa,.faa,.gz", multiple=False)
fasta_output = widgets.Output()


def _on_fasta_upload(_change):
    global input_args
    if not fasta_upload.value:
        return
    with fasta_output:
        clear_output()
        fname = list(fasta_upload.value.keys())[0]
        content = fasta_upload.value[fname]["content"]
        Path(fname).write_bytes(content)
        if fname.endswith(".gz"):
            import gzip
            extracted = fname[:-3]
            with gzip.open(fname, "rb") as gz:
                Path(extracted).write_bytes(gz.read())
            fname = extracted
        n = sum(1 for line in open(fname) if line.startswith(">"))
        print(f"Ready: {fname} ({n} sequences)")
        print("Select embedder(s) below, then proceed to Generate.")
        input_args = {"type": "fasta", "path": fname}
        embedder_section.layout.display = ""


fasta_upload.observe(_on_fasta_upload, names="value")
tab_fasta = widgets.VBox([
    widgets.HTML(
        "<b>Upload FASTA sequences</b><br>"
        "<i>Embeddings will be computed via the "
        '<a href="https://biocentral.cloud" target="_blank">Biocentral API</a></i>'
    ),
    fasta_upload,
    fasta_output,
])

# --- Tab 4: UniProt Query ---
query_input = widgets.Text(
    value="", placeholder='(family:globin) AND (reviewed:true)',
    description="Query:", style={"description_width": "initial"}, layout={"width": "100%"},
)
query_btn = widgets.Button(description="Set Query", button_style="success")
query_output = widgets.Output()


def _on_query_set(_b):
    global input_args
    with query_output:
        clear_output()
        q = query_input.value.strip()
        if not q:
            print("Enter a query.")
            return
        print(f"Query set: {q}")
        print("Select embedder(s) below, then proceed to Generate.")
        input_args = {"type": "query", "query": q}
        embedder_section.layout.display = ""


query_btn.on_click(_on_query_set)
tab_query = widgets.VBox([
    widgets.HTML("<b>Fetch proteins from UniProt</b>"),
    widgets.HBox([query_input, query_btn]),
    widgets.HTML(
        "<i>Examples:</i><br>"
        + "<br>".join(f"<code>{q}</code>" for q in _QUERY_EXAMPLES)
        + '<br><a href="https://www.uniprot.org/help/query-fields" target="_blank">Query syntax</a>'
    ),
    query_output,
])

# --- Show/hide embedder section based on active tab ---
tabs = widgets.Tab(children=[tab_example, tab_h5, tab_fasta, tab_query])
tabs.set_title(0, "Example Dataset")
tabs.set_title(1, "Upload H5")
tabs.set_title(2, "Upload FASTA")
tabs.set_title(3, "UniProt Query")


def _on_tab_change(change):
    idx = change["new"]
    if idx in (2, 3):  # FASTA or Query
        embedder_section.layout.display = ""
    else:
        embedder_section.layout.display = "none"


tabs.observe(_on_tab_change, names="selected_index")

display(widgets.VBox([tabs, embedder_section]))

# Auto-select the first example so users can just press through
_select_example(example_dropdown.value)

In [ ]:
# @title 3. Generate & Download {display-mode: "form"}
# @markdown Configure methods and annotations, then click Generate.
# @markdown Processing time depends on dataset size and selected methods.

from ipywidgets import (
    HTML,
    Button,
    Checkbox,
    Dropdown,
    FileUpload,
    FloatSlider,
    GridBox,
    HBox,
    IntSlider,
    Output,
    ToggleButton,
    VBox,
)

METHODS = ["PCA", "UMAP", "t-SNE", "MDS", "PaCMAP", "LocalMAP"]
ANNOTATIONS = {
    "UniProt": ["annotation_score", "cc_subcellular_location", "ec", "fragment", "go_bp", "go_cc", "go_mf", "keyword", "length", "protein_existence", "protein_families", "reviewed", "xref_pdb"],
    "InterPro": ["cath", "cdd", "panther", "pfam", "pfam_clan", "prints", "prosite", "signal_peptide", "smart", "superfamily"],
    "Taxonomy": ["root", "domain", "kingdom", "phylum", "class", "order", "family", "genus", "species"],
    "TED Domains": ["ted_domains"],
    "Biocentral": ["predicted_subcellular_location", "predicted_membrane", "predicted_signal_peptide", "predicted_transmembrane"],
}
ANNOTATION_DEFAULTS = {"ec", "keyword", "length", "protein_families", "reviewed"}

# --- Methods ---
method_toggles = {}
for m in METHODS:
    sel = m in {"PCA", "UMAP"}
    method_toggles[m] = ToggleButton(value=sel, description=m, button_style="info" if sel else "", layout={"width": "auto", "height": "32px"})

# --- DR Parameters ---
_s = {"description_width": "110px"}
_l = {"width": "100%"}
_g = {"border": "1px solid #666", "padding": "6px 10px", "margin": "2px", "flex": "1 1 220px", "overflow": "hidden"}

param_groups = []
w_nn = IntSlider(value=25, min=2, max=500, description="n_neighbors:", style=_s, layout=_l)
param_groups.append((VBox([HTML("<b>UMAP / PaCMAP / LocalMAP</b>"), w_nn], layout=_g), {"UMAP", "PaCMAP", "LocalMAP"}))
w_md = FloatSlider(value=0.4, min=0.0, max=0.99, step=0.01, description="min_dist:", style=_s, layout=_l)
param_groups.append((VBox([HTML("<b>UMAP</b>"), w_md], layout=_g), {"UMAP"}))
w_mn = FloatSlider(value=0.5, min=0.0, max=1.0, step=0.1, description="mn_ratio:", style=_s, layout=_l)
w_fp = FloatSlider(value=2.0, min=0.0, max=5.0, step=0.1, description="fp_ratio:", style=_s, layout=_l)
param_groups.append((VBox([HTML("<b>PaCMAP / LocalMAP</b>"), w_mn, w_fp], layout=_g), {"PaCMAP", "LocalMAP"}))
w_pp = FloatSlider(value=30.0, min=5.0, max=500.0, step=1.0, description="perplexity:", style=_s, layout=_l)
w_lr = FloatSlider(value=200.0, min=1.0, max=1000.0, step=10.0, description="learning_rate:", style=_s, layout=_l)
param_groups.append((VBox([HTML("<b>t-SNE</b>"), w_pp, w_lr], layout=_g), {"t-SNE"}))

no_params = HTML("<i>PCA/MDS have no adjustable parameters.</i>")
pw = {"n_neighbors": w_nn, "min_dist": w_md, "perplexity": w_pp, "learning_rate": w_lr, "mn_ratio": w_mn, "fp_ratio": w_fp}
param_grid = HBox([g for g, _ in param_groups], layout={"flex_flow": "row wrap", "gap": "6px", "width": "100%"})


def _update_params(_c=None):
    sel = {m for m, t in method_toggles.items() if t.value}
    any_vis = False
    for gw, req in param_groups:
        vis = bool(req & sel)
        gw.layout.display = "" if vis else "none"
        if vis:
            any_vis = True
    no_params.layout.display = "" if not any_vis else "none"


for tb in method_toggles.values():
    def _toggle(c):
        c["owner"].button_style = "info" if c["new"] else ""
        _update_params()
    tb.observe(_toggle, names="value")
_update_params()

# --- Annotations ---
ann_preset = Dropdown(
    options=["Default (recommended)", "All annotations", "Custom..."],
    value="Default (recommended)",
    description="Annotations:",
    style={"description_width": "initial"},
)
ann_cbs = {}
ann_cols = []
for cat, names in ANNOTATIONS.items():
    cbs = {n: Checkbox(value=n in ANNOTATION_DEFAULTS, description=n, indent=False, layout={"width": "auto"}) for n in names}
    ann_cbs[cat] = cbs
    grid = GridBox(list(cbs.values()), layout={"grid_template_columns": "repeat(2, auto)", "grid_gap": "0 6px"})
    ann_cols.append(VBox([HTML(f"<b>{cat}</b>"), grid], layout={"flex": "1 1 200px", "border": "1px solid #666", "padding": "6px", "margin": "2px"}))
custom_ann = HBox(ann_cols, layout={"flex_flow": "row wrap"})
custom_ann.layout.display = "none"
ann_preset.observe(lambda c: setattr(custom_ann.layout, "display", "" if c["new"] == "Custom..." else "none"), names="value")

# --- Custom CSV upload ---
csv_upload = FileUpload(accept=".csv,.tsv", multiple=False, description="Choose CSV/TSV")


def _get_ann():
    p = ann_preset.value
    if p == "Default (recommended)":
        return ["default"]
    if p == "All annotations":
        return ["all"]
    sel = [n for cbs in ann_cbs.values() for n, cb in cbs.items() if cb.value]
    return sel if sel else ["default"]


def _save_csv():
    """Save uploaded CSV to disk and return the path, or None."""
    if not csv_upload.value:
        return None
    fname = list(csv_upload.value.keys())[0]
    content = csv_upload.value[fname]["content"]
    Path(fname).write_bytes(content)
    return fname


# --- Generate ---
gen_btn = Button(description="Generate", button_style="primary", icon="play", layout={"width": "180px", "height": "40px"})
gen_out = Output()


def _on_gen(_b):
    with gen_out:
        clear_output()
        if not input_args:
            print("Select input data in cell above first.")
            return
        method_strs = [m.lower().replace("-", "") + "2" for m, t in method_toggles.items() if t.value]
        method_specs = parse_methods_arg(method_strs)
        if not method_specs:
            print("Select at least one method.")
            return

        ann = _get_ann() or []
        csv_path = _save_csv()
        if csv_path:
            ann.append(csv_path)
        if not ann:
            ann = None

        inp = input_args

        out_dir = Path("output")
        out_dir.mkdir(exist_ok=True)
        cache_dir = out_dir / "tmp"
        cache_dir.mkdir(exist_ok=True)
        output_path = out_dir / "data.parquetbundle"

        step_html = HTML(value="<b>Step 1/4: Loading embeddings...</b>")
        display(step_html)

        try:
            t0 = _time.time()
            embedding_sets = []

            if inp["type"] == "query":
                embs = _get_selected_embedders()
                if not embs:
                    print("Select at least one embedder.")
                    return
                step_html.value = "<b>Step 1/6: Downloading FASTA...</b>"
                fasta_cache = cache_dir / "sequences.fasta"
                if fasta_cache.exists() and fasta_cache.stat().st_size > 0:
                    from protspace.data.loaders.query import (
                        extract_identifiers_from_fasta,
                    )
                    headers = extract_identifiers_from_fasta(fasta_cache)
                    fasta_path = fasta_cache
                    print(f"Using cached FASTA ({len(headers):,} sequences)")
                else:
                    headers, fasta_path = query_uniprot(inp["query"], save_to=fasta_cache)
                if not headers:
                    print(f"No sequences found for query: {inp['query']}")
                    return
                from protspace.data.embedding.biocentral import EmbedConfig
                for emb_name in embs:
                    step_html.value = f"<b>Step 2/6: Computing {emb_name} embeddings...</b>"
                    emb_set = embed_fasta(
                        fasta_path, emb_name,
                        embed_config=EmbedConfig(),
                        embedding_cache=cache_dir / f"{emb_name}.h5",
                    )
                    emb_set.fasta_path = fasta_path
                    embedding_sets.append(emb_set)
            elif inp["type"] == "fasta":
                embs = _get_selected_embedders()
                if not embs:
                    print("Select at least one embedder.")
                    return
                from protspace.data.embedding.biocentral import EmbedConfig
                for emb_name in embs:
                    step_html.value = f"<b>Step 1/5: Computing {emb_name} embeddings...</b>"
                    emb_set = embed_fasta(
                        Path(inp["path"]), emb_name,
                        embed_config=EmbedConfig(),
                        embedding_cache=cache_dir / f"{emb_name}.h5",
                    )
                    emb_set.fasta_path = Path(inp["path"])
                    embedding_sets.append(emb_set)
            else:
                h5_path = Path(inp["path"])
                name_override = inp.get("name")
                emb_set = load_h5([h5_path], name_override=name_override)
                embedding_sets.append(emb_set)

            n_proteins = len(embedding_sets[0].headers)

            # Build pipeline with caching enabled
            reducer_params = ReducerParams(
                n_neighbors=pw["n_neighbors"].value,
                min_dist=pw["min_dist"].value,
                perplexity=pw["perplexity"].value,
                learning_rate=pw["learning_rate"].value,
                mn_ratio=pw["mn_ratio"].value,
                fp_ratio=pw["fp_ratio"].value,
            )
            config = PipelineConfig(
                methods=method_specs,
                output_path=output_path,
                bundled=True,
                keep_tmp=True,
                intermediate_dir=cache_dir,
                annotations=ann,
                reducer_params=reducer_params,
            )
            pipeline = ReductionPipeline(config)

            # Step 2: Annotations (cached after first run)
            step_html.value = "<b>Step 2/4: Fetching annotations...</b>"
            metadata = pipeline._fetch_annotations(embedding_sets[0].headers)

            # Step 3: Dimensionality reduction
            step_html.value = "<b>Step 3/4: Reducing dimensions...</b>"

            # Build full metadata
            all_headers = embedding_sets[0].headers
            full_metadata = _pd.DataFrame({"identifier": all_headers})
            if len(metadata.columns) > 1:
                metadata = metadata.astype(str)
                id_col = metadata.columns[0]
                if id_col != "identifier":
                    metadata = metadata.rename(columns={id_col: "identifier"})
                full_metadata = full_metadata.merge(
                    metadata.drop_duplicates("identifier"),
                    on="identifier",
                    how="left",
                )
            metadata = full_metadata

            all_reductions = pipeline._run_reductions(embedding_sets)

            # Step 4: Bundle & download
            step_html.value = "<b>Step 4/4: Bundling & downloading...</b>"
            output = pipeline.base.create_output(metadata, all_reductions, all_headers)
            pipeline.base.save_output(output, output_path, bundled=True)

            kb = output_path.stat().st_size / 1024
            total_time = _time.time() - t0
            step_html.value = f"<b>Done! ({kb:.0f} KB, {total_time:.0f}s)</b>"
            print(f"Processed {n_proteins} proteins with {len(method_specs)} method(s)")
            print("\nUpload at: https://protspace.app/explore")
            files.download(str(output_path))

        except Exception as e:
            step_html.value = "<b>Failed</b>"
            print(f"\nError: {e}")
            print("\nTroubleshooting:")
            print("- Check that protein IDs are UniProt accessions (e.g., P12345)")
            print("- Try setting Annotations to 'None' to skip annotation fetching")


gen_btn.on_click(_on_gen)

display(VBox([
    HTML("<h4>Methods</h4>"),
    HBox(list(method_toggles.values()), layout={"flex_flow": "row wrap", "gap": "6px"}),
    no_params,
    param_grid,
    HTML("<h4>Annotations</h4>"),
    HBox([ann_preset, csv_upload], layout={"gap": "10px", "align_items": "center"}),
    HTML("<p><i>Optional: upload a CSV/TSV with custom annotations (first column = protein IDs).</i></p>"),
    custom_ann,
    gen_btn,
    HTML("<p><i>Processing time depends on dataset size and selected methods.</i></p>"),
    gen_out,
]))

## What's Next

- **Explore your data** at [protspace.app/explore](https://protspace.app/explore)
- **Pre-built examples** (no processing needed): [download parquetbundles](https://github.com/tsenoner/protspace/releases/tag/examples)
- **Custom pLM embeddings**: use the [Embeddings notebook](https://colab.research.google.com/github/tsenoner/protspace/blob/main/notebooks/ClickThrough_GenerateEmbeddings.ipynb) to generate HDF5 files with 12+ protein language models
- **CLI reference**: [docs/cli.md](https://github.com/tsenoner/protspace/blob/main/docs/cli.md)
- **Annotation reference**: [docs/annotations.md](https://github.com/tsenoner/protspace/blob/main/docs/annotations.md)
